# Taak 8: doorvoer

Energie-evaluaties per seconde per model, op dezelfde 4000 dataset-posities. Baseline en student zijn dezelfde code (`energy_fast`) met andere parameterarrays, dus even snel. MACE-tijd komt uit taak 6 (joltik, V100, all-atom CH4 in de cel van 424 atomen, 0,19 s per punt).

In [1]:
import sys
sys.path.insert(0, "../src")
import time
import numpy as np
from energy import load_framework, load_force_field
from gcmc import CH4_EPS, CH4_SIGMA
from fit import element_indices, baseline_params, params_to_pairs, student_energies

positions, symbols, L, masses = load_framework("../data/IRMOF-1.cif")
lj_params = load_force_field("../data/force_field/generic_dreiding_uff_mixing_rules.def")
element_index = element_indices(symbols)
params_baseline = baseline_params(lj_params, (CH4_EPS, CH4_SIGMA))
params_student = np.loadtxt("../results/student_params.csv", delimiter=",", comments="#")[:, 2]   # kolom 'weighted'

data = np.genfromtxt("../data/mace_dataset_irmof1_ch4.csv", delimiter=",", names=True)
dataset_positions = np.column_stack([data["x"], data["y"], data["z"]])
N = len(dataset_positions)

/Users/kasperhuysentruyt/Documents/leven /Cusp/cusp-mlff-gcmc/.venv/lib/python3.14/site-packages/ase/io/cif.py:411: UserWarning: crystal system 'cubic' is not interpreted for space group Spacegroup(225, setting=1). This may result in wrong setting!
  warnings.warn(


In [2]:
def time_model(params, repeats=5):
    times = []
    for _ in range(repeats):
        t0 = time.time()
        student_energies(params, dataset_positions, positions, element_index, L)
        times.append(time.time() - t0)
    return np.median(times) / N   # s per evaluatie, mediaan over herhalingen

t_baseline = time_model(params_baseline)
t_student = time_model(params_student)
t_mace = 0.188   # s per punt, taak 6, job 40881994 op joltik (V100), mace_mp medium + D3, float64

rows = [("baseline DREIDING/UFF, energy_fast", "CPU (Apple M-serie), united atom", t_baseline),
        ("student LJ, energy_fast", "CPU (Apple M-serie), united atom", t_student),
        ("MACE-MP-0 medium + D3", "GPU (V100), all-atom CH4 + 424 roosteratomen", t_mace)]
print(f"{'model':36s} {'hardware':46s} {'s / evaluatie':>14s} {'evaluaties / s':>15s} {'t.o.v. MACE':>12s}")
for name, hw, t in rows:
    print(f"{name:36s} {hw:46s} {t:14.2e} {1 / t:15.0f} {t_mace / t:12.0f}x")

print("\nMarkdown voor de README:")
print("| model | hardware | s / evaluatie | evaluaties / s | sneller dan MACE |")
print("|---|---|---|---|---|")
for name, hw, t in rows:
    print(f"| {name} | {hw} | {t:.1e} | {1 / t:,.0f} | {t_mace / t:,.0f}x |")

model                                hardware                                        s / evaluatie  evaluaties / s  t.o.v. MACE
baseline DREIDING/UFF, energy_fast   CPU (Apple M-serie), united atom                     3.79e-05           26390         4961x
student LJ, energy_fast              CPU (Apple M-serie), united atom                     3.67e-05           27244         5122x
MACE-MP-0 medium + D3                GPU (V100), all-atom CH4 + 424 roosteratomen         1.88e-01               5            1x

Markdown voor de README:
| model | hardware | s / evaluatie | evaluaties / s | sneller dan MACE |
|---|---|---|---|---|
| baseline DREIDING/UFF, energy_fast | CPU (Apple M-serie), united atom | 3.8e-05 | 26,390 | 4,961x |
| student LJ, energy_fast | CPU (Apple M-serie), united atom | 3.7e-05 | 27,244 | 5,122x |
| MACE-MP-0 medium + D3 | GPU (V100), all-atom CH4 + 424 roosteratomen | 1.9e-01 | 5 | 1x |


## Kanttekeningen

- Appels en peren, bewust: een LJ-som op één CPU-kern tegenover een equivariant netwerk op een GPU, en één united-atom punt tegenover 429 atomen. Dat is precies de vergelijking die ertoe doet, want in GCMC is de energie van één proefmolecuul de binnenste lus.
- Een GCMC-isotherm uit taak 4 kostte ~120.000 stappen à 1 tot 2 evaluaties per drukpunt. Met de student is dat seconden per drukpunt; met MACE zou één drukpunt ~6 tot 12 uur GPU-tijd kosten, en de hele isotherm van zeven punten enkele dagen.
- Nog een factor 10 tot 100 is te halen door de roosterenergie vooraf op een 3D-grid te zetten en trilineair te interpoleren (Frenkel & Smit appendix I, zo doet RASPA het). Niet gedaan; staat in de README onder next steps.